# RL Tactical Football - Colab Setup

This notebook installs dependencies, runs training, and exports artifacts (checkpoints + frames).

**Options:**
- Set `REPO_URL` to your GitHub repo URL (or leave blank if you uploaded the folder).
- Toggle `RUN_TRAIN` to control training.

In [ ]:
# @title Config
REPO_URL = "https://github.com/crxxom/TacticalFootball.git"
PROJECT_DIR = "/content/RL_tacticalfootball"
RUN_TRAIN = True
PYTHON = "python"

In [ ]:
# Check GPU
!nvidia-smi || true

In [ ]:
import os
import subprocess

if REPO_URL:
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL} {PROJECT_DIR}
    else:
        print("Project directory already exists.")
else:
    if not os.path.exists(PROJECT_DIR):
        raise FileNotFoundError(f"{PROJECT_DIR} not found. Upload your folder or set REPO_URL.")

%cd {PROJECT_DIR}

In [ ]:
# Install dependencies
!{PYTHON} -m pip install --upgrade pip
!{PYTHON} -m pip install -r requirements.txt

## Ray Dashboard (Colab)
Run this *before* training. The dashboard becomes available once `train.py` starts Ray. Use the URL printed from the log.

In [ ]:
# Install cloudflared for a public tunnel to the Ray dashboard (port 8265)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Start tunnel (the URL appears in the log)
!nohup cloudflared tunnel --url http://localhost:8265 > /content/cloudflared.log 2>&1 &

# Show the latest log lines (look for a https://*.trycloudflare.com URL)
!tail -n 5 /content/cloudflared.log


In [ ]:
if RUN_TRAIN:
    !{PYTHON} train.py
else:
    print("Skipping training.")

In [ ]:
# Zip and download artifacts
!zip -r checkpoints.zip runs/checkpoints || true
!zip -r frames.zip runs/frames || true
from google.colab import files
if os.path.exists("checkpoints.zip"):
    files.download("checkpoints.zip")
if os.path.exists("frames.zip"):
    files.download("frames.zip")